In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 — Gold: Temperature Alerts
# MAGIC Reads Silver as a stream, flags any window/device combination running hot.

# COMMAND ----------

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

# COMMAND ----------

from iot_pipeline.config import SILVER_TABLE, GOLD_TABLE, CHECKPOINT_BASE, TEMP_ALERT_THRESHOLD
from iot_pipeline.transforms import detect_alerts

# COMMAND ----------

checkpoint_path = CHECKPOINT_BASE + "gold/"

silver_stream = spark.readStream.table(SILVER_TABLE)
alerts_stream = detect_alerts(silver_stream, TEMP_ALERT_THRESHOLD)

def upsert_to_gold(micro_batch_df, batch_id):
    spark = micro_batch_df.sparkSession
    if not spark.catalog.tableExists(GOLD_TABLE):
        micro_batch_df.write.format("delta").saveAsTable(GOLD_TABLE)
    else:
        micro_batch_df.createOrReplaceTempView("updates")
        spark.sql(f"""
            MERGE INTO {GOLD_TABLE} AS target
            USING updates AS source
            ON target.window = source.window AND target.device_id = source.device_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

query = (alerts_stream.writeStream
    .outputMode("update")
    .foreachBatch(upsert_to_gold)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

query.awaitTermination()

# COMMAND ----------

# MAGIC %md ## Validation

# COMMAND ----------

count = spark.table(GOLD_TABLE).count()
print(f"✅ {GOLD_TABLE}: {count} rows")

if count > 0:
    devices = [r["device_id"] for r in spark.table(GOLD_TABLE).select("device_id").distinct().collect()]
    print(f"   Alerting device(s): {devices}")
print("Expected: 3 rows, all device_id = 'sensor_3' (the seed data's built-in anomaly window).")

In [0]:
%restart_python